# Testing provenance

## Table of content

- [Create toy dataframes](#create-toy-dataframes)
- [Tests with data_provenance_enabled](#tests-with-data_provenance_enabled)
    - [Select](#select)
    - [Join](#join)
    - [Where](#where)
    - [Aggregation](#aggregation)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#%pip install pyspark


In [ ]:
from wringlet import (
    data_provenance_enabled, 
    data_provenance_session_builder,
    # add_provenance_column, 
    # remove_provenance_column
)


In [ ]:
from pyspark.sql import SparkSession


active_session = SparkSession.getActiveSession()
if active_session is not None:
    active_session.stop()

spark = (
    # SparkSession
    # .builder
    data_provenance_session_builder("semiwhy")
    .appName("data-provenance-notebook")
    .getOrCreate()
)


In [ ]:
print(spark.conf.get("spark.provenance.enabled", "false"))

with data_provenance_enabled(spark):
    print(spark.conf.get("spark.provenance.enabled", "false"))

print(spark.conf.get("spark.provenance.enabled", "false"))

## Create toy dataframes

In [ ]:
from datetime import date
df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("C", date(2026, 1, 17), 80.0, 60),
    ("F", date(2026, 1, 16), 50.0, 70)
], ["product", "date", "price", "quantity"]
)
df.printSchema()

df.show(truncate=False)

In [ ]:
df2 = spark.createDataFrame([
    ("A", "Bike"),
    ("B", "Handball"),
    ("C", "Bike"),
    ("D", "Handball"),
    ("E", "Running")
],["letter","labell"]
)

df2.printSchema()
df2.show(truncate=False)



In [ ]:
from datetime import date

dft = spark.createDataFrame([
    ("A", date(2026, 1, 14), 10.0, 50),
    ("D", date(2026, 1, 15), 20.0, 70),
    ("C", date(2026, 1, 17), 80.0, 30),
    ("B", date(2026, 1, 18), 100.0, 10),
    ("F", date(2026, 1, 18), 80.0, 50),
    ("C", date(2026, 1, 15), 80.0, 60),
    ("F", date(2026, 1, 17), 75.0, 70)
], ["product", "date", "price", "quantity"]
)

dft.printSchema()
dft.show(truncate=False)

## Tests with data_provenance_enabled
### Select

In [ ]:
df.select("*").show(truncate=False)

In [ ]:
dft.createOrReplaceTempView("sales")
with data_provenance_enabled(spark, df, df2, "sales") as (df_bis, df2_bis, dft_bis):
    print(df_bis.show(truncate=False))
    print(df2_bis.show(truncate=False))
    print(spark.table(dft_bis).show(truncate=False))


In [ ]:
dft.createOrReplaceTempView("sales")
with data_provenance_enabled(spark, df, df2, "sales") as (df_bis, df2_bis, dft_bis):
    print(df_bis.select("*").filter(df_bis["product"] == "A").show(truncate=False), end="\n\n")
    print(df2_bis.select("*").filter(df2_bis["letter"] == "B").show(truncate=False), end="\n\n")
    print(spark.table(dft_bis).filter(spark.table(dft_bis)["product"] == "C").show(truncate=False), end="\n\n")



In [ ]:
with data_provenance_enabled(spark, df) as (df_bis):
    df3 = df_bis.select("*")
df3.show(truncate=False)

In [ ]:
with data_provenance_enabled(spark, df2) as (df2_bis):
    df4 = df2_bis.select("letter")
    df4.show(truncate=False)


In [ ]:
# Some tags for df2 are the same as those for df
df.createOrReplaceTempView("sales")
with data_provenance_enabled(spark, "sales"):
    res = spark.sql("select product from sales")
res.show(truncate=False)

In [ ]:
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark,"category"):
    res2 = spark.sql("select letter from category")
res2.show(truncate=False)

### Join

In [ ]:
df.createOrReplaceTempView("sales")
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark, "sales", "category") as (df_sales, df_category):
    res = spark.sql(
        f"select * from {df_sales} s join {df_category} c on s.product = c.letter"
    )
res.show(truncate=False)

In [ ]:
# Only the provenance tags of df is taken in account
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df5 = df_bis.select("*").join(df2_bis, df_bis.product == df2_bis.letter)

df5.show(truncate=False)

In [ ]:
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df6 = df_bis.join(df2_bis, df_bis.product == df2_bis.letter, "outer").select("product")

df6.show(truncate=False)

In [ ]:
# Same here, only the provenance tags of df are taken in account
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df7 = df_bis.select("*").join(df2_bis, df_bis.product==df2_bis.letter, "outer")
df7.show(truncate=False)

### Where

In [ ]:
with data_provenance_enabled(spark, df) as (df_bis):
    df8 = df_bis.select("*").filter("price > 10")
df8.show(truncate=False)

In [ ]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales") as (df_sales):
    res = spark.sql("select product, price from sales where price>10").show(truncate=False)

res

In [ ]:
# With select statement
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df9 = df_bis.select("*").filter("price > 10").sort("price").join(df2_bis, df_bis.product == df2_bis.letter)
df9.show(truncate=False)

In [ ]:
# Without select statement
with data_provenance_enabled(spark, df, df2) as (df_bis, df2_bis):
    df10 = df_bis.filter("price > 10").sort("price").join(df2_bis, df_bis.product == df2_bis.letter)
df10.show(truncate=False)

### Aggregation

In [ ]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales"):
    res = spark.sql("select sum(quantity) from sales group by product").show(truncate=False)
res

In [ ]:
with data_provenance_enabled(spark, df) as (df_bis):
    df11 = df_bis.groupBy("product").agg({"quantity": "sum"})
df11.show(truncate=False)

In [ ]:
with data_provenance_enabled(spark, df) as (df_bis):
    df12 = df_bis.withColumn("revenue", df_bis["quantity"] * df_bis["price"]) \
            .groupBy("product") \
            .agg({"revenue": "sum"})
df12.show(truncate=False)

In [ ]:
with data_provenance_enabled(spark, df) as (df_bis):
    df13 = df_bis.groupBy("product").agg({"quantity": "sum"})
df13.show(truncate=False)

In [ ]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales") :
    res = spark.sql("select distinct product from sales").show(truncate=False)

res

In [ ]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales"
) :
    res = spark.sql("select product from sales group by product").show(truncate=False)
res

In [ ]:
with data_provenance_enabled(spark, df, dft) as (df_bis, dft_bis):
    df14 = df_bis.select("product").distinct().union(dft_bis.select("product").distinct())
df14.show(truncate=False)

In [ ]:
# The provenance tags of df and dft are not always different
# The distinct operation does not work as expected, it does not always create new tags for the distinct values
# and it can reuse the same tags for the same values in df and dft

print(spark.conf.get("spark.provenance.enabled", "false"))
df15 = df_bis.select("product").distinct().union(dft_bis.select("product").distinct())
print(spark.conf.get("spark.provenance.enabled", "false"))
df15.show(truncate=False)

In [ ]:
df.createOrReplaceTempView("sales1")
dft.createOrReplaceTempView("sales2")

with data_provenance_enabled(spark, "sales1", "sales2") as (df_sales1, df_sales2):
    res = spark.sql("select s1.product from sales1 as s1 union select s2.product from sales2 as s2")
res.show(truncate=False)

In [ ]:
# Same exemple without provenance
df.createOrReplaceTempView("sales1")
dft.createOrReplaceTempView("sales2")
spark.sql("select s1.product from sales1 as s1 union select s2.product from sales2 as s2").show(truncate=False)

In [ ]:
df.createOrReplaceTempView("sales")
spark.sql("select * from sales").show(truncate=False)

In [ ]:
with data_provenance_enabled(spark, df) as (df_bis):
    # 1. Pipeline
    df8 = df_bis.select("*").filter("price > 10")
    
    # 2. On extrait les sources minimales DEPUIS df_bis tant que les tags sont actifs !
    minimal_sources = df8.get_minimal_sources([df_bis])

# 3. On affiche le résultat (en dehors du bloc avec succès)
initial_rows = minimal_sources[0]
initial_rows.show(truncate=False)